In [1]:
import xarray as xr
import numpy as np
import json
import pandas as pd
import os
import glob

def generate_boxplot_data():
    print("GENERATOR BACKEND: BOXPLOT STATISTICS ENGINE")

    # 1. KONFIGURASI
    start_year = 1979
    end_year = 2024
    data_folder = '.'
    output_filename = 'boxplot_data_final.json'
    
    configs = {
        'wind': {'u_pat': '*u_component*', 'v_pat': '*v_component*', 'vars': ['u10', 'v10', 'u', 'v']},
        'swh': {'pat': '*significant_height*', 'vars': ['swh', 'significant_height']},
        'mwp': {'pat': '*mean_wave_period*', 'vars': ['mwp', 'mean_wave_period']}
    }

    final_json = {}

    for var_name, cfg in configs.items():
        print(f"\nProcessing Variable: {var_name.upper()}")
        yearly_list = []
        lats, lons = None, None

        for year in range(start_year, end_year + 1):
            try:
                if var_name == 'wind':
                    u_f = glob.glob(os.path.join(data_folder, f"{cfg['u_pat']}*{year}*.nc"))
                    v_f = glob.glob(os.path.join(data_folder, f"{cfg['v_pat']}*{year}*.nc"))
                    if not u_f or not v_f: continue
                    ds_u = xr.open_dataset(u_f[0]); ds_v = xr.open_dataset(v_f[0])
                    if 'valid_time' in ds_u.coords: ds_u = ds_u.rename({'valid_time': 'time'})
                    if 'valid_time' in ds_v.coords: ds_v = ds_v.rename({'valid_time': 'time'})
                    v_u = next(v for v in cfg['vars'] if v in ds_u)
                    v_v = next(v for v in cfg['vars'] if v in ds_v)
                    data_hourly = np.sqrt(ds_u[v_u]**2 + ds_v[v_v]**2)
                    ds_u.close(); ds_v.close()
                else:
                    f = glob.glob(os.path.join(data_folder, f"{cfg['pat']}*{year}*.nc"))
                    if not f: continue
                    ds = xr.open_dataset(f[0])
                    if 'valid_time' in ds.coords: ds = ds.rename({'valid_time': 'time'})
                    v_target = next(v for v in cfg['vars'] if v in ds)
                    data_hourly = ds[v_target]
                
                if lats is None:
                    lats, lons = data_hourly.latitude.values, data_hourly.longitude.values

                # Hitung 5-number summary per bulan
                monthly_stats = data_hourly.resample(time='1MS').quantile([0, 0.25, 0.5, 0.75, 1.0], dim='time')
                yearly_list.append(monthly_stats)
                
                print(f"   [{year}] Stats Extracted", end='\r')
                if var_name != 'wind': ds.close()
            except Exception as e:
                print(f" Error {year}: {e}")

        if not yearly_list: continue

        # Gabungkan dan paksa urutan dimensi: (time, quantile, lat, lon)
        ds_all_ts = xr.concat(yearly_list, dim='time').transpose("time", "quantile", "latitude", "longitude")
        
        # Hitung Clim (12) dan Seas (4) - transpose juga untuk keamanan
        ds_clim = ds_all_ts.groupby('time.month').mean(dim='time').transpose("month", "quantile", "latitude", "longitude")
        ds_seas = ds_all_ts.groupby('time.season').mean(dim='time').transpose("season", "quantile", "latitude", "longitude")

        def clean_list_2d(arr_2d):
            """Memastikan output array 2D (Time/Month/Season, 5)"""
            return [[round(float(v), 3) if (v is not None and np.isfinite(v)) else None for v in row] for row in arr_2d]

        # 1. Hitung Area Average (Sumbu Waktu tetap di urutan pertama)
        aa_ts = ds_all_ts.mean(dim=['latitude', 'longitude'], skipna=True).values
        aa_clim = ds_clim.mean(dim=['latitude', 'longitude'], skipna=True).values
        aa_seas = ds_seas.mean(dim=['latitude', 'longitude'], skipna=True).values

        var_output = {
            "metadata": {
                "labels_ts": [pd.to_datetime(t).strftime('%Y-%m') for t in ds_all_ts.time.values],
                "labels_clim": ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
                "labels_seas": ["DJF", "MAM", "JJA", "SON"]
            },
            "area_average": {
                "ts": clean_list_2d(aa_ts),
                "clim": clean_list_2d(aa_clim),
                "seas": clean_list_2d(aa_seas)
            },
            "grid_data": {}
        }

        # 2. Hitung Grid Data
        print(f"Saving Grid Stats for {var_name}...")
        for i in range(len(lats)):
            for j in range(len(lons)):
                key = f"{i},{j}"
                # Ambil data: (time, quantile)
                ts_vals = ds_all_ts.values[:, :, i, j]
                
                if np.isnan(ts_vals).all():
                    continue
                
                var_output["grid_data"][key] = {
                    "ts": clean_list_2d(ts_vals),
                    "clim": clean_list_2d(ds_clim.values[:, :, i, j]),
                    "seas": clean_list_2d(ds_seas.values[:, :, i, j])
                }
        
        final_json[var_name] = var_output

    final_json["metadata"] = {
        "latitudes": lats.tolist(),
        "longitudes": lons.tolist(),
        "stats_order": ["min", "q1", "median", "q3", "max"]
    }

    with open(output_filename, 'w') as f:
        json.dump(final_json, f)
    
    print(f"\n✅ SELESAI! File '{output_filename}' diperbaiki dan siap digunakan.")

generate_boxplot_data()

GENERATOR BACKEND: BOXPLOT STATISTICS ENGINE

Processing Variable: WIND
Saving Grid Stats for wind...

Processing Variable: SWH


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


...

C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
C:\Users\acer\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


Saving Grid Stats for mwp...

✅ SELESAI! File 'boxplot_data_final.json' diperbaiki dan siap digunakan.
